# Encoding Diagnostics

Audit the existing E2.5 comparison rather than repeat model/encoding selection. All saved methods must share raw and encoded schemas, model parameters, split sizes, and low-cardinality one-hot encoding. Refit the native method once to verify identical metrics, category dtypes, effective categorical-column indices, persisted categories, and actual categorical (`==`) tree splits. The official E2/E3 results and encoding remain unchanged.

This fair comparison is scoped to core BLOCK/STREET, not auxiliary categories or the composite identifier. Profile all six high-cardinality columns for training support, unseen validation categories and missingness. Ordinal codes for nominal labels impose an artificial order; categorical splits are genuinely different. Frequency can conflate categories with equal frequencies. Neither flaw implies a particular method must perform best on this split.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').is_dir()
sys.path.insert(0, str(PROJECT_ROOT))
from experiments.feature_engineering.utils.diagnostics import *
from IPython.display import display, Image


In [2]:
comparison, profiles = encoding_diagnostics()
display(comparison[['encoding','RMSE','MAE','R2','Mean Error']])
display(profiles)
display(json.loads((artifact('encoding_audit.json')).read_text()))

/Users/wangyufan/Desktop/master/CS5228/project/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,encoding,RMSE,MAE,R2,Mean Error
0,ordinal,479.496863,356.508969,0.470103,-45.033072
1,frequency,479.805553,357.295635,0.469421,-47.312233
2,native,482.634963,359.682289,0.463145,-47.311319


,feature,train_categories,validation_categories,train_missing,validation_missing,unknown_validation_rows,unknown_validation_pct,median_train_rows_per_category,train_categories_with_under_20_rows,formal_encoding,used_in_encoding_comparison
0,BLOCK,2699,2585,0,0,50,0.158318,19.0,1350,ordinal_numeric,True
1,STREET,574,573,0,0,4,0.012665,137.0,43,ordinal_numeric,True
2,SUBZONE,156,156,0,0,0,0.000000,656.0,2,ordinal_numeric,False
3,mrt_name,136,135,0,0,0,0.000000,513.5,3,ordinal_numeric,False
4,mall_name,85,85,0,0,0,0.000000,982.0,2,ordinal_numeric,False
5,composite_block_id,9324,8366,0,0,131,0.414793,10.0,7639,ordinal_numeric,False


{'fair_comparison': True,
 'categorical_dtypes': {'BLOCK': 'category', 'STREET': 'category'},
 'categorical_feature_argument': ['BLOCK', 'STREET'],
 'effective_categorical_column': [3, 4],
 'pandas_category_counts': [2699, 574],
 'categorical_split_count': 7893,
 'categorical_split_features': ['BLOCK', 'STREET'],
 'reproduced_metrics': {'RMSE': 482.63496333358154,
  'MAE': 359.68228916393923,
  'R2': 0.46314457473841253,
  'Mean Error': -47.31131923730856},
 'training_seconds': 2.7029961249791086,
 'block_labels_mapping_to_multiple_addresses': 1462,
 'block_labels_total': 2699,
 'scope': 'Comparison covers BLOCK/STREET on E2.5 only, not auxiliary categories or composite_block_id.',
 'recommendation': 'Keep official encoding unchanged. Native is implemented correctly but was worse on the tested core.'}